In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from datetime import datetime
import json
import hashlib

IMAGE_SIZE = 64
BATCH_SIZE = 128
NUM_EPOCHS = 30
LEARNING_RATE = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(IMAGE_SIZE, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_dataset = datasets.CIFAR100(root="./data", train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR100(root="./data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
model = EmbeddingCNN(embedding_dim=256, num_classes=100).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output, _ = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)
    
    scheduler.step()
    return total_loss / len(loader), correct / total

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output, _ = model(data)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    return correct / total

In [ ]:
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_acc = evaluate(model, test_loader, device)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
    
    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"./checkpoint_epoch_{epoch+1}.pth"
        torch.save({
            'model_state_dict': model.state_dict(),
            'embedding_dim': 256,
            'class_names': train_dataset.classes,
            'test_accuracy': test_acc,
            'epoch': epoch + 1,
            'checkpoint_mtime': datetime.now().timestamp(),
            'checkpoint_md5': hashlib.md5(open('./embedding_model.pth', 'rb').read()).hexdigest() if os.path.exists('./embedding_model.pth') else '',
        }, checkpoint_path)
        print(f"Saved checkpoint to {checkpoint_path}")

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'embedding_dim': 256,
    'class_names': train_dataset.classes,
    'test_accuracy': test_acc,
    'checkpoint_mtime': datetime.now().timestamp(),
    'checkpoint_md5': hashlib.md5(open('./embedding_model.pth', 'rb').read()).hexdigest() if os.path.exists('./embedding_model.pth') else '',
}, 'embedding_model.pth')

print(f"\nTraining complete! Final test accuracy: {test_acc:.4f}")
print(f"Checkpoint saved to embedding_model.pth")